In [ ]:
!pip install numpy ipympl librosa scipy

In [ ]:
import numpy as np
import librosa as lr
from numpy import pi
ε = 1e-10
from scipy import signal
import random
import matplotlib.pyplot as plt
%matplotlib inline

# Вариант 6

### Задание

1. Ознакомиться с теоретической частью.

1. Для тестовых музыкальных файлов реализовать мел-спектраграмму:

    - [x] реализовать алгоритм вычисления мел-спектраграммы
  
    - [x] реализовать мел-спектраграмму с помощью готовых библиотек
  
    - объяснить полученный результат
  
1. Для тестовых музыкальных файлов вычислить спектральные признаки аудиосигнала:

    - самостоятельно:
  
        - [x] частота пересечения нуля
      
        - [x] спектральная ширина
     
    - с помощью библиотек:
  
        - [x] спектральный центроид
     
        - [x] спектральный спад
     
        - [x] цветность

    - объяснить полученный результат
  
1. [x] Написать функцию, которая будет смешивать чистый голос и шум по `SNR = 0,3..15 дБ`.

1. Сравнить методы оценки качества звука:

    - самостоятельно:
  
        - [ ] SNR
     
        - [ ] SDR
     
    - с помощью библиотек:
  
        - [ ] SI-SDR
     
        - [ ] PESQ
     
        - [ ] NISQA
     
        - [ ] DNSMOS
     
    - [ ] вывести в виде таблицы:
  
      тестовый файл | объективные оценки (SNR, SDR, etc.) | субъективная оценка
      ---           |---                                  |---
  
1. Прогнать через шумоподавление:

    - [ ] установить модель шумоподавления (DeepFilterNet2 или более актуальную)
  
    - [ ] повторить тесты из п.5
      

In [ ]:
SNR = range(0, 15, 3)

FIGSIZE=(20, 4)

In [ ]:
# samples, sr = lr.load('mambo_no_5-lou_bega.wav', sr=None)
samples, sr = lr.load('africa-toto.wav', sr=None)
samples = samples[sr*30:sr*40] # taking from 30th to 40th seconds

sample_times = np.linspace(0, len(samples)/sr, len(samples))

def plot_samples_overtime():
    plt.plot(sample_times, samples)

plt.figure(figsize=FIGSIZE)
plot_samples_overtime()
plt.show()

In [ ]:
def make_noise(signal, ratios_db):
    try: ratios_db[0]
    except: ratios_db = (ratios_db,)
        
    pow = lambda x: np.mean(x**2)

    signal_pow = pow(signal)
    
    noise = np.array([random.uniform(-1.0, 1.0) for _ in signal])
    noise_pow = pow(noise)

    nice = [] # mouse -> mice | noise -> nice
    for ratio_db in ratios_db:
        target_noise_pow = signal_pow / 10**(ratio_db/10)
        scaled_noise = noise * (target_noise_pow/noise_pow)**0.5
        assert(abs(ratio_db - 10*np.log10(signal_pow / pow(scaled_noise))) < 0.001)
        nice.append(scaled_noise)
    return nice

noises = make_noise(samples, SNR)

### МЕЛ-Спектрограмма

In [ ]:
def plot_lr_mel_spectr(samples, mel_count=128):
    mel_spectr = lr.feature.melspectrogram(y=samples, sr=sr, n_mels=mel_count)
    mel_spectr_db = lr.power_to_db(mel_spectr, top_db=None)
    
    lr.display.specshow(mel_spectr_db, sr=sr, x_axis='time', y_axis='mel', cmap="magma", )
    plt.colorbar(format="%+.f dB")
    plt.tight_layout()
    
plt.figure(figsize=FIGSIZE)
plot_lr_mel_spectr(samples)
plt.show()

In [ ]:
def plot_our_mel_spectr(samples, n_mels=128):
    def mel(hz): return 2595 * np.log10(1 + hz/700)
    def unmel(mel): return 700 * (10**(mel/2595) - 1)
    def db(x): return 10 * np.log10(x + ε) # fix bug when `x` can be `-0.0`

    def mel_filters(count: int, freqs):
        f_min_mel = mel(min(freqs))
        f_max_mel = mel(max(freqs))
        f_mel_step = (f_max_mel - f_min_mel) / (count+1)

        filters, mel_freqs = [], []
        for f_mel in np.arange(f_min_mel + f_mel_step, f_max_mel, f_mel_step):
            f_mid, f_start, f_end = unmel(f_mel), unmel(f_mel-f_mel_step), unmel(f_mel+f_mel_step)
            mid_sample, start_sample, end_sample = np.abs(freqs-f_mid).argmin(), np.abs(freqs-f_start).argmin(), np.abs(freqs-f_end).argmin()

            tri_leftwidth, tri_rightwidth = max(1, mid_sample-start_sample), max(1, end_sample-mid_sample)
            tri = np.concatenate((
                np.linspace(0, 1, num=tri_leftwidth),
                np.linspace(1, 0, num=tri_rightwidth)
            ))

            filter = np.pad(tri, pad_width=(start_sample, len(freqs)-start_sample-tri_leftwidth-tri_rightwidth), constant_values=0)
            filters.append(filter)
            mel_freqs.append(f_mel)

        return np.array(filters), np.array(mel_freqs)
        
    # https://www.youtube.com/watch?v=-Yxj3yfvY-4
    freqs, times, stft = signal.stft(samples, sr, nperseg=1024) # TODO? probably implement ourselves
    ampls = np.abs(stft)**2
    
    filters, mel_freqs = mel_filters(n_mels, freqs)
    mel_ampls = np.dot(filters, ampls)

    mel_ampls = db(mel_ampls)

    plt.pcolormesh(times, mel_freqs, mel_ampls, cmap="magma")
    plt.ylabel('Mel')
    plt.xlabel('Time')
    plt.colorbar(format="%+.f dB")
    plt.tight_layout()
    
plt.figure(figsize=FIGSIZE)
plot_our_mel_spectr(samples)
plt.show()

### Спектральные признаки

#### Спектральная Ширина

In [ ]:
def spectral_centroid(freqs, times, ampls):
    M = len(freqs)
    return [
        sum(ampls[k][l] * freqs[k] for k in range(M))
         / (sum(ampls[k][l] for k in range(M)) + ε)
    for l in range(len(times))]

def plot_spectral_width(samples):
    def spectral_width(freqs, times, ampls):
        centroid = spectral_centroid(freqs, times, ampls)
        
        M = len(freqs)
        return [
            np.sqrt(
                sum(ampls[k][l] * (freqs[k]-centroid[l])**2 for k in range(M))
                 / (sum(ampls[k][l] for k in range(M)) + ε)
            )
        for l in range(len(times))]

    freqs, times, stft = signal.stft(samples, sr, nperseg=1024)
    ampls = np.abs(stft)
    sw = spectral_width(freqs, times, ampls)
    
    plot_lr_mel_spectr(samples)
    plt.plot(times, sw, color="yellow")
    
plt.figure(figsize=FIGSIZE)
plot_spectral_width(samples)
plt.show()

#### Частота Пересечения Нуля

In [ ]:
def plot_zero_crossing_rate(samples):
    def zero_crossing_rate(x, frame_size:int=8192, hop:int|None=None):
        hop = hop or frame_size//2
        
        return [
            1/frame_size * np.sum([np.sign(x[n]) != np.sign(x[n-1]) for n in range(i, i + frame_size)])
            for i in range(0, len(x)-frame_size, hop)
        ]

    zcr = zero_crossing_rate(samples)
    
    plot_samples_overtime()
    plt.plot(np.linspace(0, len(samples)/sr, len(zcr)), zcr, color="red")

plt.figure(figsize=FIGSIZE)
plot_zero_crossing_rate(samples)
plt.show()

#### Спектральный Центроид

In [ ]:
def plot_spectral_centroid(samples):
    sc = lr.feature.spectral_centroid(y=samples, sr=sr)[0]
    times = lr.times_like(sc, sr=sr)
    
    plot_lr_mel_spectr(samples)
    plt.plot(times, sc, color="yellow")
    
plt.figure(figsize=FIGSIZE)
plot_spectral_centroid(samples)
plt.show()

#### Спектральный Спад

In [ ]:
def plot_spectral_centroid(samples):
    srol = lr.feature.spectral_rolloff(y=samples, sr=sr)[0]
    times = lr.times_like(srol, sr=sr)

    plot_lr_mel_spectr(samples)
    plt.plot(times, srol, color="yellow")
    
plt.figure(figsize=FIGSIZE)
plot_spectral_centroid(samples)
plt.show()

#### Цветность

In [ ]:
def plot_spectral_centroid(samples):
    chroma = lr.feature.chroma_cqt(y=samples, sr=sr)
    
    lr.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma', cmap="magma")
    plt.colorbar(format="%+.f dB")
    plt.tight_layout()
    
plt.figure(figsize=FIGSIZE)
plot_spectral_centroid(samples)
plt.show()